# 02 · League baseline models

**Run first:** `python -m src.models` · **Spec:** [`docs/analysis_spec.md`](../docs/analysis_spec.md) sections 7.1–7.5

The baseline answers one question: *what does a play in this situation normally produce?* It never sees an Auburn or Florida outcome. Every table below is read from files written by `src/models.py`.

Order of operations:
1. Tune on 2021–2024.
2. Test once on 2025.
3. Apply the frozen selection rule.
4. Refit on 2021–2025.
5. Check the refit baseline against other 2026 games.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)
T = config.TABLES_DIR
selection = json.loads((config.OUTPUTS_DIR / "models" / "model_selection.json").read_text())

## 1. Tuning (leave one season out, 2021–2024)

Losses are mean squared error for PPA and log loss for explosive plays. Tuning also ran for the alternate explosive target (robustness check 5), using the model family chosen for the primary explosive target.

In [2]:
tuning = pd.read_csv(T / "model_tuning.csv")
display(tuning[["target", "family", "setting", "cv_loss"]].round(6))

,target,family,setting,cv_loss
0,value,linear,penalty=1,1.646433
1,value,linear,penalty=10,1.646427
2,value,linear,penalty=100,1.646443
3,value,linear,penalty=1000,1.646769
4,value,linear,penalty=10000,1.648736
5,value,boosted,"depth=4, leaf=light",1.608933
6,value,boosted,"depth=6, leaf=light",1.608796
7,value,boosted,"depth=4, leaf=heavy",1.609401
8,value,boosted,"depth=6, leaf=heavy",1.609781
9,explosive,linear,penalty=0.01,0.234931


## 2. The one-time 2025 test

The table covers every candidate plus the naive baseline (the 2021–2024 average). The "no spread" rows answer the plan's question of whether the pregame spread helps. `group_bias` is the play-weighted average absolute error across down × distance × field position × run/pass groups, which is the situational bias that would otherwise leak into team residuals.

In [3]:
test = pd.read_csv(T / "model_test_2025.csv")
value_cols = ["candidate", "mae", "rmse", "r2", "mean_error", "group_bias"]
explosive_cols = ["candidate", "brier", "log_loss", "roc_auc", "average_precision", "observed_over_expected", "calibration_slope", "group_bias"]
print("PPA per play"); display(test[test.target == "value"][value_cols].round(4))
print("Explosive plays (20+ yards)"); display(test[test.target == "explosive"][explosive_cols].round(4))

PPA per play


,candidate,mae,rmse,r2,mean_error,group_bias
0,naive,0.9444,1.3034,-0.0001,-0.0109,0.2019
1,linear,0.9291,1.2708,0.0493,-0.0148,0.0778
2,linear (no spread),0.9335,1.2746,0.0436,-0.0144,0.0770
3,boosted,0.9250,1.2556,0.0719,-0.0199,0.0282
4,boosted (no spread),0.9292,1.2590,0.0669,-0.0199,0.0276


Explosive plays (20+ yards)


,candidate,brier,log_loss,roc_auc,average_precision,observed_over_expected,calibration_slope,group_bias
5,naive,0.0634,0.2484,NaN,0.0680,0.9798,NaN,0.0364
6,linear,0.0616,0.2321,0.6923,0.1230,0.9767,0.9704,0.0055
7,linear (no spread),0.0617,0.2328,0.6860,0.1200,0.9766,0.9693,0.0056
8,boosted,0.0614,0.2305,0.6998,0.1283,0.9658,0.9811,0.0047
9,boosted (no spread),0.0615,0.2311,0.6951,0.1260,0.9636,0.9800,0.0048


## 3. Selection rule and gates

- **Selection:** the boosted model replaces the transparent model only with at least a 1% gain in MAE (PPA) or Brier score (explosive). The 95% interval is game-clustered.
- **Gate 2:** a model beats naive on RMSE/MAE (PPA) or Brier/log loss/ROC-AUC (explosive), *and* cuts group bias by at least 50%.
- **Gate 3:** observed/expected within 0.95–1.05 and calibration slope within 0.85–1.15.

In [4]:
rows = []
for target in ("value", "explosive"):
    s = selection[target]
    g = s["boosted_relative_gain"]
    rows.append({
        "target": target, "boosted gain": f"{g['estimate']:.2%} (95% CI {g['ci95'][0]:.2%} to {g['ci95'][1]:.2%}) in {g['metric']}",
        "chosen": s["chosen"], "chosen settings": s["spec"]["params"],
        "group bias cut vs naive": f"{s['group_bias_reduction']:.0%}", "gate 2: beats naive": s["beats_naive"],
        "gate 3: calibrated": s.get("calibrated", "n/a"),
    })
display(pd.DataFrame(rows).set_index("target"))

,boosted gain,chosen,chosen settings,group bias cut vs naive,gate 2: beats naive,gate 3: calibrated
target,,,,,,
value,0.44% (95% CI 0.31% to 0.56%) in mae,linear,{'penalty': 10.0},61%,True,n/a
explosive,0.28% (95% CI 0.20% to 0.36%) in brier,linear,{'penalty': 1.0},85%,True,True


**Reading the result**

- **Boosting helps, but not enough to clear the bar.** It cuts MAE by 0.44% and Brier score by 0.28%. Both intervals exclude zero but sit well below 1%, so the frozen rule keeps ridge regression (PPA) and logistic regression (explosive).
- **Both chosen models clear gates 2 and 3.** PPA group bias falls 61% relative to naive. The explosive model reaches ROC-AUC 0.69, observed/expected 0.98, and calibration slope 0.97.
- **The spread helps modestly.** Dropping it raises ridge MAE from 0.929 to 0.934 and lowers explosive ROC-AUC from 0.692 to 0.686.
- **Ridge leaves more situational bias than boosting** (0.078 vs 0.028 PPA per play across groups). That's why check 10 (the boosted baseline must agree) was added before any team residual existed. It can only remove findings.
- **Low R² is expected.** Pre-snap context explains about 5% of play-to-play PPA variance, because most of a play's value is decided after the snap. The baseline's job is to remove *systematic* situational differences, which the group-bias numbers measure.

## 4. Where the transparent baseline misses, by situation (2025)

Mean error (observed minus predicted) for the chosen model and for the naive baseline.

In [5]:
groups = pd.read_csv(T / "model_test_groups_2025.csv")
for target in ("value", "explosive"):
    g = groups[(groups.target == target) & (groups.dimension != "week")]
    wide = g.pivot_table(index=["dimension", "group"], columns="candidate", values="mean_error").round(4)
    wide["plays"] = g.groupby(["dimension", "group"]).plays.first()
    print(target); display(wide)
weeks = groups[(groups.dimension == "week") & (groups.candidate != "naive")].pivot_table(index="group", columns="target", values="mean_error").round(4)
print("Mean error by 2025 week, chosen models"); display(weeks.T)

value


candidate                        linear   naive  plays
dimension    group                                    
distance_bin long 8+            -0.0103 -0.0597  80872
             medium 4-7         -0.0193  0.0637  23164
             short 1-3          -0.0310  0.1289  15915
down_bin     1st                -0.0129 -0.1500  52446
             2nd                -0.0178 -0.0351  39171
             3rd/4th            -0.0140  0.2802  28334
field_bin    midfield to opp 21 -0.0329 -0.2154  36723
             own territory      -0.0005  0.0794  66217
             red zone           -0.0313  0.0791  17011
play_family  pass               -0.0098  0.0265  59235
             run                -0.0196 -0.0474  60716

explosive


candidate                        linear   naive  plays
dimension    group                                    
distance_bin long 8+            -0.0011  0.0049  80872
             medium 4-7         -0.0020 -0.0069  23164
             short 1-3          -0.0038 -0.0255  15915
down_bin     1st                -0.0017  0.0009  52446
             2nd                -0.0036 -0.0074  39171
             3rd/4th             0.0012  0.0027  28334
field_bin    midfield to opp 21 -0.0035  0.0141  36723
             own territory      -0.0009  0.0054  66217
             red zone           -0.0005 -0.0614  17011
play_family  pass               -0.0022  0.0295  59235
             run                -0.0010 -0.0315  60716

Mean error by 2025 week, chosen models


group,01,02,03,04,05,06,07,08,09,10,11,12,13,14,15,16,postseason
target,,,,,,,,,,,,,,,,,
explosive,-0.0078,0.0012,-0.0023,0.0008,-0.0015,-0.0050,-0.0015,-0.0012,-0.0043,-0.0000,-0.0015,0.0015,-0.0002,0.0043,-0.0091,0.0133,-0.0055
value,-0.0387,-0.0114,0.0069,0.0066,-0.0036,-0.0073,-0.0142,-0.0010,-0.0340,-0.0172,-0.0356,-0.0177,-0.0069,-0.0077,-0.0905,-0.0373,-0.0264


## 5. Explosive-play calibration on 2025

In [6]:
display(pd.read_csv(T / "calibration_2025.csv").round(4))

,decile,plays,mean_predicted,observed_rate,target,candidate
0,1,11996,0.0017,0.0021,explosive,linear
1,2,11995,0.0248,0.0218,explosive,linear
2,3,11995,0.0384,0.0399,explosive,linear
3,4,11995,0.0453,0.0449,explosive,linear
4,5,11995,0.0531,0.0518,explosive,linear
5,6,11995,0.0673,0.0677,explosive,linear
6,7,11995,0.0887,0.0855,explosive,linear
7,8,11995,0.1042,0.1020,explosive,linear
8,9,11995,0.1197,0.1214,explosive,linear
9,10,11995,0.1528,0.1426,explosive,linear


## 6. The 2026 environment check

The final baseline (refit on 2021–2025) is applied to every other 2026 FBS play through the cutoff, excluding the four matchup games. A correction applies only if the miss exceeds the frozen thresholds: ±0.03 PPA per play with the interval excluding zero, or observed/expected outside 0.90–1.10 with the interval excluding one. For contrast, the 2025 Weeks 1–2 row shows the same statistic for last season's opening weeks under the 2021–2024 model, which never saw them.

In [7]:
env = pd.read_csv(T / "environment_check_2026.csv")
display(env[["model", "target", "plays", "games", "statistic", "estimate", "ci_low", "ci_high", "triggered", "correction"]].round(4))

,model,target,plays,games,statistic,estimate,ci_low,ci_high,triggered,correction
0,final,value,22640,181,mean residual,-0.0159,-0.0340,0.0031,False,0.0
1,final,explosive,22640,181,observed / expected,0.9423,0.8955,0.9886,False,0.0
2,final,explosive_alt,22640,181,observed / expected,0.9590,0.9280,0.9888,False,0.0
3,sensitivity,value,22640,181,mean residual,-0.0134,-0.0314,0.0055,False,0.0
4,sensitivity,explosive,22640,181,observed / expected,0.9454,0.8987,0.9909,False,0.0
5,sensitivity,explosive_alt,22640,181,observed / expected,0.9613,0.9304,0.9916,False,0.0
6,boosted_final,value,22640,181,mean residual,-0.0206,-0.0394,-0.0010,False,0.0
7,boosted_final,explosive,22640,181,observed / expected,0.9275,0.8814,0.9719,False,0.0
8,2025 Weeks 1-2 benchmark (2021-2024 model),value,22883,179,mean residual,-0.0259,-0.0446,-0.0060,NaN,NaN
9,2025 Weeks 1-2 benchmark (2021-2024 model),explosive,22883,179,observed / expected,0.9489,0.8952,1.0033,NaN,NaN


**No correction was triggered.**
- Early 2026 runs slightly below the baseline: −0.016 PPA per play and 94% of expected explosive plays.
- 2025's opening two regular-season weeks looked the same (−0.026 and 95%), so this is an early-season pattern, not a 2026-specific shift.
- Early-season offenses sit slightly below a full-season baseline. That pulls both matchup components (Auburn's offense residual and the residual of offenses facing Florida) slightly negative, and so pulls the edge slightly against Auburn.
- The pull is small. A −0.016 league shift, shrunk by the 2–12% factors used for these cells (notebook 03), moves an edge by roughly −0.001 PPA per play, about a tenth the size of the robust findings. It stays uncorrected under the frozen threshold and is disclosed as a limitation.